<a href="https://colab.research.google.com/github/blu-e7/Tekra-2026-Select-nama_tim/blob/main/TekraPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import warnings
import joblib
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, roc_auc_score, f1_score,
    confusion_matrix, roc_curve
)

from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTENC
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


df = pd.read_csv('Student Mental health.csv')

df.columns = [
    'Timestamp', 'Gender', 'Age', 'Course',
    'Year_of_Study', 'CGPA', 'Marital_Status',
    'Depression', 'Anxiety', 'Panic_Attack', 'Treatment'
]
df.drop('Timestamp', axis=1, inplace=True)

df['Age'] = df['Age'].fillna(df['Age'].median())

for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].astype(str).str.strip()

df['Course']        = df['Course'].str.lower()
df['Year_of_Study'] = df['Year_of_Study'].str.lower()

course_rumpun = {
    'bcs': 'computing', 'bit': 'computing', 'koe': 'computing',
    'engineering': 'engineering', 'engine': 'engineering',
    'biomedical science': 'science', 'biomed': 'science',
    'mathemathics': 'science', 'maths': 'science',
    'pendidikan islam': 'islamic_studies', 'islamic education': 'islamic_studies',
    'kirkhs': 'islamic_studies', 'irkhs': 'islamic_studies',
    'psychology': 'social_science', 'human resources': 'social_science',
    'benl': 'humanities', 'enm': 'humanities',
    'laws': 'law', 'law': 'law',
    'accounting': 'business', 'business administration': 'business',
}
df['Course_group'] = df['Course'].map(lambda x: course_rumpun.get(x, 'others'))

print("Dataset awal:", df.shape)
print("Missing values:", df.isnull().sum().sum())

Dataset awal: (101, 11)
Missing values: 0


In [4]:
df['Target_Risk'] = (
    (df['Depression'] == 'Yes') |
    (df['Anxiety']    == 'Yes') |
    (df['Panic_Attack'] == 'Yes')
).astype(int)

print("\nDistribusi Target_Risk:")
print(df['Target_Risk'].value_counts())



Distribusi Target_Risk:
Target_Risk
1    64
0    37
Name: count, dtype: int64


In [5]:
np.random.seed(42)
n = len(df)

att = np.random.normal(loc=84 - 5 * df['Target_Risk'].values, scale=12, size=n)
df['Attendance_Rate'] = np.clip(att, 40, 100).round(1)

lam_late = np.where(df['Target_Risk'] == 1, 3.0, 2.0)
df['Late_Submissions'] = np.clip(
    np.random.poisson(lam=lam_late, size=n), 0, 12
)

p_ukt = np.where(df['Target_Risk'] == 1, 0.30, 0.15)
df['UKT_Delay'] = np.array([np.random.binomial(1, p) for p in p_ukt])

print("\nVerifikasi korelasi fitur sintetik vs target (harus < |0.5|):")
for col in ['Attendance_Rate', 'Late_Submissions', 'UKT_Delay']:
    corr = df[col].corr(df['Target_Risk'])
    status = "✓" if abs(corr) < 0.5 else "⚠ terlalu tinggi"
    print(f"  {col:<22}: {corr:+.4f}  {status}")




Verifikasi korelasi fitur sintetik vs target (harus < |0.5|):
  Attendance_Rate       : -0.0895  ✓
  Late_Submissions      : +0.2912  ✓
  UKT_Delay             : +0.2414  ✓


In [7]:
df_enc = df.copy()

df_enc['Gender']         = df_enc['Gender'].map({'Female': 0, 'Male': 1})
df_enc['Marital_Status'] = df_enc['Marital_Status'].map({'No': 0, 'Yes': 1})

cgpa_order = {
    '0 - 1.99': 1, '2.00 - 2.49': 2,
    '2.50 - 2.99': 3, '3.00 - 3.49': 4, '3.50 - 4.00': 5,
}
df_enc['CGPA_ord'] = df_enc['CGPA'].map(cgpa_order).fillna(4)

year_order = {'year 1': 1, 'year 2': 2, 'year 3': 3, 'year 4': 4}
df_enc['Year_ord'] = df_enc['Year_of_Study'].map(year_order).fillna(1)

le_course = LabelEncoder()
df_enc['Course_enc'] = le_course.fit_transform(df_enc['Course_group'])

In [8]:
FEATURES = [
    'Gender',
    'Age',
    'CGPA_ord',
    'Year_ord',
    'Marital_Status',
    'Course_enc',
    'Attendance_Rate',    # dari SIAM (simulasi)
    'Late_Submissions',   # dari sistem pengumpulan tugas (simulasi)
    'UKT_Delay',          # dari sistem keuangan (simulasi)
]
TARGET = 'Target_Risk'

X = df_enc[FEATURES]
y = df_enc[TARGET]

print(f"\nFitur input : {len(FEATURES)} variabel")
print(f"Distribusi  : {y.value_counts().to_dict()}")


Fitur input : 9 variabel
Distribusi  : {1: 64, 0: 37}


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_indices = [0, 4, 5, 8]  # Gender, Marital_Status, Course_enc, UKT_Delay

smote_nc = SMOTENC(categorical_features=cat_indices, random_state=42)
X_train_bal, y_train_bal = smote_nc.fit_resample(X_train, y_train)

print(f"\nTrain sebelum SMOTENC : {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Train setelah SMOTENC : {dict(zip(*np.unique(y_train_bal, return_counts=True)))}")
print(f"Test (tidak disentuh) : {dict(zip(*np.unique(y_test, return_counts=True)))}")


Train sebelum SMOTENC : {np.int64(0): np.int64(29), np.int64(1): np.int64(51)}
Train setelah SMOTENC : {np.int64(0): np.int64(51), np.int64(1): np.int64(51)}
Test (tidak disentuh) : {np.int64(0): np.int64(8), np.int64(1): np.int64(13)}


In [10]:
xgb = XGBClassifier(eval_metric='logloss', random_state=42)

param_grid = {
    'n_estimators':  [50, 100],
    'max_depth':     [2, 3],        # dangkal untuk data kecil → cegah overfitting
    'learning_rate': [0.05, 0.1],
    'subsample':     [0.8, 1.0],    # tambahan: regularisasi via row sampling
}

cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    xgb, param_grid,
    cv=cv_strat, scoring='roc_auc',
    n_jobs=-1
)
grid_search.fit(X_train_bal, y_train_bal)

best_xgb = grid_search.best_estimator_

print(f"\nParameter terbaik : {grid_search.best_params_}")
print(f"CV AUC terbaik    : {grid_search.best_score_:.4f}")



Parameter terbaik : {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 50, 'subsample': 1.0}
CV AUC terbaik    : 0.7928


In [11]:
y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:, 1]

print("\n=== EVALUASI MODEL ===")
print(classification_report(y_test, y_pred, target_names=['Aman', 'Berisiko']))
print(f"Test AUC  : {roc_auc_score(y_test, y_prob):.4f}")
print(f"Test F1   : {f1_score(y_test, y_pred):.4f}")

cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:")
print(f"  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")
tn, fp, fn, tp = cm.ravel()
print(f"\n  Recall/Sensitivity : {tp/(tp+fn):.2f}  ← mahasiswa berisiko yang terdeteksi")
print(f"  Specificity        : {tn/(tn+fp):.2f}  ← mahasiswa aman yang tidak di-alarm")

# Threshold optimal via Youden's J
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
opt_idx = np.argmax(tpr - fpr)
opt_threshold = thresholds[opt_idx]
y_pred_opt = (y_prob >= opt_threshold).astype(int)

print(f"\nThreshold optimal (Youden's J) : {opt_threshold:.3f}")
print(f"F1 dengan threshold optimal    : {f1_score(y_test, y_pred_opt):.4f}")



=== EVALUASI MODEL ===
              precision    recall  f1-score   support

        Aman       0.40      0.25      0.31         8
    Berisiko       0.62      0.77      0.69        13

    accuracy                           0.57        21
   macro avg       0.51      0.51      0.50        21
weighted avg       0.54      0.57      0.54        21

Test AUC  : 0.5769
Test F1   : 0.6897

Confusion Matrix:
  TN=2  FP=6
  FN=3  TP=10

  Recall/Sensitivity : 0.77  ← mahasiswa berisiko yang terdeteksi
  Specificity        : 0.25  ← mahasiswa aman yang tidak di-alarm

Threshold optimal (Youden's J) : 0.531
F1 dengan threshold optimal    : 0.7407


In [14]:
explainer   = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test)

# Global feature importance (summary plot)
plt.figure()
shap.summary_plot(
    shap_values, X_test,
    feature_names=FEATURES,
    show=False
)
plt.title("SHAP Feature Importance – Indikator Risiko Psikologis Mahasiswa")
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.close()
print("save to shap_summary.png")

# Mean absolute SHAP per fitur
mean_shap = pd.DataFrame({
    'Fitur': FEATURES,
    'Mean |SHAP|': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean |SHAP|', ascending=False)

print("\n[Top Feature Importance via SHAP]")
for _, row in mean_shap.iterrows():
    bar = '█' * int(row['Mean |SHAP|'] * 120)
    print(f"  {row['Fitur']:<22} {row['Mean |SHAP|']:.4f}  {bar}")


save to shap_summary.png

[Top Feature Importance via SHAP]
  Late_Submissions       0.4360  ████████████████████████████████████████████████████
  UKT_Delay              0.3738  ████████████████████████████████████████████
  Marital_Status         0.2731  ████████████████████████████████
  Course_enc             0.2026  ████████████████████████
  Age                    0.1278  ███████████████
  Attendance_Rate        0.0330  ███
  Year_ord               0.0323  ███
  Gender                 0.0000  
  CGPA_ord               0.0000  


In [15]:
def risk_level(prob: float, threshold: float) -> tuple:
    high = threshold + (1 - threshold) * 0.4
    low  = threshold * 0.5
    if prob >= high:
        return "TINGGI",  "Segera jadwalkan sesi konseling dengan psikolog kampus."
    elif prob >= low:
        return "SEDANG",  "Pantau lebih dekat; rekomendasikan konsultasi awal."
    else:
        return "RENDAH",  "Kondisi relatif aman — lanjutkan pemantauan rutin."


def predict_risk(
    gender: str,           # 'Female' / 'Male'
    age: float,
    cgpa: str,             # '3.00 - 3.49', dst.
    year_of_study: str,    # 'year 1' – 'year 4'
    marital_status: str,   # 'Yes' / 'No'
    course: str,           # nama jurusan (bebas, akan di-map)
    attendance_rate: float,   # % kehadiran dari SIAM
    late_submissions: int,    # jumlah tugas terlambat
    ukt_delay: int,           # 1 = ada keterlambatan UKT, 0 = tidak
) -> dict:
    g_enc  = 1 if gender == 'Male' else 0
    cgpa_v = cgpa_order.get(cgpa, 4)
    yr_v   = year_order.get(year_of_study.lower(), 1)
    mar_v  = 1 if marital_status == 'Yes' else 0

    cg = course_rumpun.get(course.lower().strip(), 'others')
    if cg in le_course.classes_:
        course_v = le_course.transform([cg])[0]
    else:
        course_v = le_course.transform(['others'])[0]

    row  = np.array([[g_enc, age, cgpa_v, yr_v, mar_v, course_v,
                      attendance_rate, late_submissions, ukt_delay]])
    prob = best_xgb.predict_proba(row)[0][1]
    level, action = risk_level(prob, opt_threshold)

    return {
        'Probabilitas Risiko': f"{prob*100:.1f}%",
        'Level Risiko'       : level,
        'Rekomendasi'        : action,
    }


# ── Contoh Inferensi ─────────────────────────────────────────
print("\n=== CONTOH SISTEM PERINGATAN DINI ===")

test_cases = [
    {
        'label': "Mahasiswi Tahun 3, kehadiran rendah, banyak tugas telat",
        'params': dict(gender='Female', age=21, cgpa='2.50 - 2.99',
                       year_of_study='year 3', marital_status='No',
                       course='engineering', attendance_rate=62.0,
                       late_submissions=7, ukt_delay=1)
    },
    {
        'label': "Mahasiswa Tahun 1, CGPA tinggi, kehadiran rajin",
        'params': dict(gender='Male', age=19, cgpa='3.50 - 4.00',
                       year_of_study='year 1', marital_status='No',
                       course='bcs', attendance_rate=95.0,
                       late_submissions=1, ukt_delay=0)
    },
    {
        'label': "Mahasiswi Tahun 4, kehadiran sedang, ada kendala UKT",
        'params': dict(gender='Female', age=23, cgpa='3.00 - 3.49',
                       year_of_study='year 4', marital_status='Yes',
                       course='psychology', attendance_rate=75.0,
                       late_submissions=4, ukt_delay=1)
    },
]

for case in test_cases:
    res = predict_risk(**case['params'])
    print(f"\n  Kasus: {case['label']}")
    for k, v in res.items():
        print(f"    {k:<25}: {v}")


=== CONTOH SISTEM PERINGATAN DINI ===

  Kasus: Mahasiswi Tahun 3, kehadiran rendah, banyak tugas telat
    Probabilitas Risiko      : 74.2%
    Level Risiko             : TINGGI
    Rekomendasi              : Segera jadwalkan sesi konseling dengan psikolog kampus.

  Kasus: Mahasiswa Tahun 1, CGPA tinggi, kehadiran rajin
    Probabilitas Risiko      : 33.0%
    Level Risiko             : SEDANG
    Rekomendasi              : Pantau lebih dekat; rekomendasikan konsultasi awal.

  Kasus: Mahasiswi Tahun 4, kehadiran sedang, ada kendala UKT
    Probabilitas Risiko      : 58.8%
    Level Risiko             : SEDANG
    Rekomendasi              : Pantau lebih dekat; rekomendasikan konsultasi awal.


In [16]:
artifacts = {
    'model'         : best_xgb,
    'label_encoder' : le_course,
    'features'      : FEATURES,
    'threshold'     : opt_threshold,
    'maps'          : {
        'cgpa'        : cgpa_order,
        'year'        : year_order,
        'course'      : course_rumpun,
    }
}
joblib.dump(artifacts, 'mental_health_ews_smartcampus.pkl')

print(f"\nPipeline EWS Smart Campus berhasil diekspor.")
print(f"  Model      : XGBoost (params: {grid_search.best_params_})")
print(f"  Threshold  : {opt_threshold:.3f}")
print(f"  Test AUC   : {roc_auc_score(y_test, y_prob):.4f}")
print(f"  File       : mental_health_ews_smartcampus.pkl")


Pipeline EWS Smart Campus berhasil diekspor.
  Model      : XGBoost (params: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 50, 'subsample': 1.0})
  Threshold  : 0.531
  Test AUC   : 0.5769
  File       : mental_health_ews_smartcampus.pkl
